# Unsloth Tam Rehber (2026, Resmi Pattern)

Bu rehber **Unsloth ekibinin resmi Colab notebook'larından** çıkarılmıştır. Kafamdan tahmin değil — gerçek pattern'ler.

Referans aldığım resmi notebooklar:
- `Gemma4_(E2B)_Text.ipynb`
- `Qwen3_(4B)_Instruct.ipynb`
- `Qwen3_(4B)_Thinking.ipynb`
- `Qwen3_5_(2B)_Vision.ipynb`

## İçindekiler

1. **Class Ailesi** — Hangi class ne zaman?
2. **`unsloth.chat_templates` Modülü** — Resmi yaklaşımın kalbi
3. **Text SFT Pattern** — `train_on_responses_only` ile
4. **Vision SFT Pattern** — `UnslothVisionDataCollator` ile
5. **Thinking Model SFT** — Reasoning data + `enable_thinking`
6. **Hyperparameter Rehberi** — Resmi default'lar
7. **Inference Pattern'leri**
8. **Save & Deploy**
9. **Hata Kataloğu**
10. **Karar Akışı + Quick Start**

## 1. Class Ailesi — Hangisi Ne Zaman?

Resmi notebook'ların gösterdiği:

| Class | Model Tipi | Kullanan Resmi Notebook |
|-------|-----------|--------------------------|
| **`FastLanguageModel`** | Text-only LLM | Qwen3 4B Instruct/Thinking |
| **`FastModel`** | Text-only LLM (yeni unified) | Gemma 4 E2B Text |
| **`FastVisionModel`** | Vision-Language Model | Qwen3.5 2B Vision |

**Önemli:** `FastLanguageModel` LEGACY DEĞİL — Qwen3 için resmi notebook hala bu class'ı kullanıyor. `FastModel` daha yeni unified API ama her ikisi de aktif.

**Pratik kural:**
- Text SFT yapacaksan: `FastLanguageModel` veya `FastModel` (her ikisi OK, model bazlı tercih)
- Vision SFT yapacaksan: **`FastVisionModel`** (zorunlu)
- Karışık (VLM ama text-only kullanım): `FastModel` daha güvenli

## Karar Akışı

```
Modelin tipi nedir?
│
├── Text-only LLM (Qwen3, Llama-3, Mistral)
│   ├── Resmi Qwen3 notebook'u takip ediyorum → FastLanguageModel
│   └── Modern unified API istiyorum → FastModel
│
├── VLM, sadece text fine-tune (Gemma 4 E2B text task)
│   └── FastModel + finetune_vision_layers=False
│
└── VLM, vision SFT (Qwen3.5 2B Vision)
    └── FastVisionModel + UnslothVisionDataCollator
```

In [ ]:
import unsloth
[c for c in dir(unsloth) if 'Fast' in c or 'Model' in c]

## 2. `unsloth.chat_templates` Modülü — Resmi Yaklaşımın Kalbi

Resmi notebook'ların hepsinde 3 fonksiyon kullanılıyor:

```python
from unsloth.chat_templates import (
    get_chat_template,           # tokenizer'a chat template uygula
    standardize_data_formats,    # dataset format normalize
    train_on_responses_only,     # completion-only mask
)
```

### 2.1 `get_chat_template`

Tokenizer'a model-spesifik chat template uygular:

```python
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-thinking")
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
```

Built-in template adları (kısmi liste):
- `gemma-4`, `gemma-3`, `gemma-2`, `gemma-1`
- `qwen3-instruct`, `qwen3-thinking`, `qwen-2.5`
- `llama-3`, `llama-3.1`, `llama-3.2`
- `mistral`, `phi-4`

### 2.2 `standardize_data_formats`

Farklı dataset formatlarını (sharegpt, alpaca, vb.) standart `conversations` formatına çevirir:

```python
from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split="train")
dataset = standardize_data_formats(dataset)
# Şimdi 'conversations' kolonu var, format normalize edilmiş
```

### 2.3 `train_on_responses_only` — KRİTİK

Bu fonksiyon **TRL'nin `assistant_only_loss=True`'sundan AYRI**. Unsloth'un kendi yöntemi, VLM kontrolünden bağımsız.

```python
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(...)  # önce trainer oluştur

# Sonra sar — instruction maskele, sadece response loss'a girsin
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",      # Qwen3 için
    response_part = "<|im_start|>assistant\n",   # Qwen3 için
)
```

### Instruction/Response Token'ları — Model-Spesifik

| Model Ailesi | `instruction_part` | `response_part` |
|--------------|---------------------|-----------------|
| **Qwen3** (Instruct/Thinking) | `<\|im_start\|>user\n` | `<\|im_start\|>assistant\n` |
| **Gemma 4** | `<\|turn>user\n` | `<\|turn>model\n` |
| **Llama 3** | `<\|start_header_id\|>user<\|end_header_id\|>\n\n` | `<\|start_header_id\|>assistant<\|end_header_id\|>\n\n` |
| **Mistral** | `[INST]` | `[/INST]` |

Tokens'ı doğrudan tokenizer'dan görebilirsin:
```python
sample_text = tokenizer.apply_chat_template(messages, tokenize=False)
print(sample_text[:200])  # Format'ı incele
```

## 3. Text SFT Pattern — Resmi Akış

Resmi `Qwen3_(4B)_Instruct.ipynb` baz alınarak:

In [ ]:
import unsloth                                              # 1. EN BAŞTA
from unsloth import FastLanguageModel
from unsloth.chat_templates import (
    get_chat_template, standardize_data_formats, train_on_responses_only,
)
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 2. Model yükle
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048,
    load_in_4bit = True,                  # QLoRA
    load_in_8bit = False,
    full_finetuning = False,
)

# 3. LoRA — KLASIK target_modules listesi (resmi notebook'larda yaygın)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,                                # 8/16/32/64/128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,                       # genelde r ile aynı
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # %30 az VRAM
    random_state = 3407,
    use_rslora = False,                    # RSLoRA opsiyonel
    loftq_config = None,                   # LoftQ opsiyonel
)

# 4. Chat template
tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-instruct")

# 5. Dataset — standardize + format
dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:1000]")
dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# 6. Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,                 # processing_class değil — resmi notebook'larda tokenizer=
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,                   # sabit step (warmup_ratio değil)
        max_steps = 60,                     # veya num_train_epochs = 1
        learning_rate = 2e-4,               # LoRA için
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,               # 0.001 — resmi default
        lr_scheduler_type = "linear",       # linear — resmi default
        seed = 3407,
        report_to = "none",
    ),
)

# 7. KRİTİK — train_on_responses_only ile masking
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

# 8. Train
trainer_stats = trainer.train()

### Masking'in Çalıştığını Doğrula

Resmi notebook'larda bu doğrulama her zaman var:

```python
# 100. örnek — full input
print(tokenizer.decode(trainer.train_dataset[100]["input_ids"]))
# Bu instruction + response gösterir

# 100. örnek — sadece label'lı kısım (mask edilmemişler)
print(tokenizer.decode([
    tokenizer.pad_token_id if x == -100 else x
    for x in trainer.train_dataset[100]["labels"]
]))
# Bu sadece response kısmını gösterir, instruction `<pad>` ile dolu
```

Beklenti: Sadece assistant cevabı görünür, user mesajı `<pad>` token'larıyla maskelenmiş.

### Gemma 4 İçin Aynı Pattern (farklı detaylar)

Resmi `Gemma4_(E2B)_Text.ipynb` baz alınarak:

```python
from unsloth import FastModel              # FastLanguageModel değil!

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,                           # auto detection
    max_seq_length = 1024,
    load_in_4bit = False,                   # 16-bit LoRA için False
    full_finetuning = False,
)

# LoRA — Gemma 4 için finetune_* flags + r=8
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,    # Text-only için kapat
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,                                  # küçük rank
    lora_alpha = 8,                         # alpha == r
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

# Chat template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")

# ... formatting + trainer ...

# train_on_responses_only Gemma 4 token'larıyla
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)
```

## 4. Vision SFT Pattern

Resmi `Qwen3_5_(2B)_Vision.ipynb` baz alınarak. **Tamamen farklı pattern** — `train_on_responses_only` YOK, `UnslothVisionDataCollator` VAR.

In [ ]:
import unsloth
from unsloth import FastVisionModel              # AYRI class
from unsloth.trainer import UnslothVisionDataCollator
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 1. Model — FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-2B",
    load_in_4bit = False,                  # 16-bit LoRA
    use_gradient_checkpointing = "unsloth",
)

# 2. LoRA — finetune_* flag'leri (vision dahil)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,     # Vision SFT için True
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    # target_modules = "all-linear",        # opsiyonel
)

# 3. Vision data — image + text content
dataset = load_dataset("unsloth/LaTeX_OCR", split="train")

instruction = "Write the LaTeX representation for this image."

def convert_to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": sample["image"]},   # PIL.Image
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["text"]},
            ]},
        ]
    }
converted_dataset = [convert_to_conversation(s) for s in dataset]

# 4. Training — KRİTİK config
FastVisionModel.for_training(model)        # Training moduna geç

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),  # ZORUNLU
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        
        # Vision SFT için ZORUNLU üçlü:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

trainer_stats = trainer.train()

## 5. Thinking Model SFT (Reasoning)

Resmi `Qwen3_(4B)_Thinking.ipynb` baz alınarak. Text SFT pattern'iyle aynı, sadece veri ve chat template farklı:

```python
# Model — Thinking variant
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Thinking-2507",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Chat template — thinking variant
tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-thinking")

# Dataset — reasoning data (math/CoT)
dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")

def generate_conversation(examples):
    return {"conversations": [
        [
            {"role": "user", "content": problem},
            {"role": "assistant", "content": solution},
        ]
        for problem, solution in zip(examples["problem"], examples["generated_solution"])
    ]}
dataset = dataset.map(generate_conversation, batched=True)
```

**Inference farklı — `enable_thinking` parametresi:**

```python
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,                  # True/False — reasoning bloklarını aç/kapat
)
```

Training'te tüm veride thinking blokları var (CoT). Inference'ta `enable_thinking=False` ile sadece final answer alabilirsin.

## 6. Hyperparameter Rehberi (Resmi Default'lar)

### LoRA Parameters

| Param | Resmi Değer | Not |
|-------|-------------|-----|
| `r` | 8 (Gemma E2B), 16, 32 (Qwen3 4B) | Model boyutuna göre |
| `lora_alpha` | r ile aynı (genelde) | r=8 → α=8, r=32 → α=32 |
| `lora_dropout` | 0 | Optimized for speed |
| `bias` | `"none"` | Optimized |
| `use_gradient_checkpointing` | `"unsloth"` | %30 daha az VRAM |
| `random_state` | 3407 | Reproducibility |
| `use_rslora` | False | RSLoRA destekleniyor (rank ≥ 64'te düşün) |
| `loftq_config` | None | LoftQ destekleniyor |

### LoRA Target Specification — İki Yöntem

**Yöntem 1 — Klasik `target_modules`** (Qwen3 notebook'larında):
```python
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
```

**Yöntem 2 — Modern `finetune_*`** (Gemma 4 ve Vision notebook'larında):
```python
finetune_vision_layers     = False,
finetune_language_layers   = True,
finetune_attention_modules = True,
finetune_mlp_modules       = True,
```

**Yöntem 3 — Tek string `"all-linear"`:**
```python
target_modules = "all-linear"   # opsiyonel, vision notebook'unda comment
```

Hepsi aynı sonucu verir genelde. `finetune_*` flag'leri **VLM'lerde tercih edilir** (vision/language ayrımı için).

### Training Parameters (SFTConfig)

| Param | Resmi Default | Notlar |
|-------|---------------|--------|
| `per_device_train_batch_size` | 2 | GPU'ya göre artır |
| `gradient_accumulation_steps` | 4 | Effective = batch × accum |
| `warmup_steps` | **5** | Sabit step, ratio değil |
| `max_steps` | 30-60 (demo) | Production: `num_train_epochs=1` |
| `learning_rate` | **2e-4** | LoRA için. Long training: 2e-5 |
| `logging_steps` | 1 | Her step log |
| `optim` | **`"adamw_8bit"`** | bitsandbytes 8-bit |
| `weight_decay` | **0.001** | (Önceden 0.01 demiştim, yanlış) |
| `lr_scheduler_type` | **`"linear"`** | (Cosine değil!) |
| `seed` | 3407 | |
| `report_to` | `"none"` | wandb/trackio için override |

### Vision SFT EK Parametreler

```python
remove_unused_columns = False
dataset_text_field = ""
dataset_kwargs = {"skip_prepare_dataset": True}
max_length = 2048
```

**Bu üçü olmadan vision data parse edilmez.**

## 7. Inference Pattern'leri

### Standart Text Inference

```python
from transformers import TextStreamer

messages = [{"role": "user", "content": "Faiz nedir?"}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,            # ZORUNLU — generation için
)

_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens = 1000,
    streamer = TextStreamer(tokenizer, skip_prompt=True),
    # Generation params modele göre değişir!
)
```

### Generation Params — Model-Spesifik Öneriler

| Model | temperature | top_p | top_k | min_p |
|-------|-------------|-------|-------|-------|
| Qwen3 (non-thinking) | 0.7 | 0.8 | 20 | — |
| Qwen3 thinking modu | farklı | — | — | — |
| Gemma 4 | 1.0 | 0.95 | 64 | — |
| Qwen3.5 Vision | 1.5 | — | — | 0.1 |

### Thinking Inference

```python
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,                 # Reasoning'i kapat
)
```

### Vision Inference

```python
messages = [{
    "role": "user",
    "content": [
        {"type": "image"},                   # placeholder
        {"type": "text", "text": "Describe this image."},
    ]
}]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

# Image tokenizer'a 1. arg olarak geçer:
inputs = tokenizer(
    image,                                    # PIL.Image first
    input_text,                              # text second
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens = 128,
    temperature = 1.5, min_p = 0.1,
)
```

## 8. Save & Deploy

Resmi notebook'larda 4 save yöntemi:

### A. LoRA Adapter
```python
model.save_pretrained("qwen_lora")
tokenizer.save_pretrained("qwen_lora")
```

### B. Hub'a Push (LoRA only)
```python
model.push_to_hub("HF_ACCOUNT/qwen_lora", token="hf_xxx")
tokenizer.push_to_hub("HF_ACCOUNT/qwen_lora", token="hf_xxx")
```

### C. Merged 16-bit (vLLM için)
```python
model.save_pretrained_merged(
    "qwen_merged",
    tokenizer,
    save_method = "merged_16bit",
)
```

### D. GGUF (Ollama / llama.cpp)
```python
# Tek quant
model.save_pretrained_gguf(
    "qwen_gguf",
    tokenizer,
    quantization_method = "q4_k_m",     # veya q8_0, f16
)

# Birden çok quant aynı anda
model.push_to_hub_gguf(
    "HF_ACCOUNT/qwen_gguf",
    tokenizer,
    quantization_method = ["q4_k_m", "q8_0", "f16"],
    token = "hf_xxx",
)
```

### LoRA Yükleme (yeni session'da)
```python
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "qwen_lora",            # Save edilmiş klasör veya HF repo
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
```

## 9. Hata Kataloğu

### Hata 1: `Unsloth: Should be imported before trl, transformers, peft`
**Sebep:** Import sırası yanlış. 
**Çözüm:** `import unsloth`'u en başa al.

### Hata 2: `KeyError: 'qwen3_5'`
**Sebep:** transformers eski. 
**Çözüm:** `transformers>=5.5` kur.

### Hata 3: `libcusparseLt.so.0` veya `libnvshmem_host.so.3`
**Sebep:** Lock upgrade sonrası CUDA dependency eksik. 
**Çözüm:** `uv pip install --reinstall nvidia-cusparselt-cu12 nvidia-nvshmem-cu12 torch`

### Hata 4: `flash-attn` warning
**Sebep:** flash-attn Python sürüm uyumsuz. 
**Çözüm:** Yok say — Xformers fallback, %10 hız kaybı.

### Hata 5: `RuntimeError: ... formatting_func`
**Sebep:** Conversational format aldı ama `formatting_prompts_func` yazılmamış. 
**Çözüm:** Resmi pattern'i takip et — `dataset.map(formatting_prompts_func)` ile text alanı oluştur.

### Hata 6: Vision SFT'de tüm labels = -100
**Sebep:** `assistant_only_loss=True` kullandın (VLM ile uyumsuz). 
**Çözüm:** `UnslothVisionDataCollator` + zorunlu üçlü config kullan, `assistant_only_loss` setleme.

### Hata 7: Vision SFT'de KeyError dataset alanları
**Sebep:** `remove_unused_columns=True` (default) — vision kolonları siliyor. 
**Çözüm:** `remove_unused_columns=False` set et.

### Hata 8: VRAM yetmiyor
**Sırasıyla dene:**
1. `load_in_4bit=True` (QLoRA)
2. `gradient_accumulation_steps` arttır, `per_device_batch_size` azalt
3. `use_gradient_checkpointing="unsloth"`
4. `max_seq_length` azalt
5. `r` azalt

## 10. Hızlı Başlangıç — Resmi Pattern Şablonu

Aşağıdaki şablon **Qwen3 4B Instruct** resmi notebook'una %95 sadık. Sen modeli ve dataset'i değiştirebilirsin.

In [ ]:
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import (
    get_chat_template, standardize_data_formats, train_on_responses_only,
)
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# ============================================================
# 1. Model
# ============================================================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048,
    load_in_4bit = True,
    full_finetuning = False,
)

# ============================================================
# 2. LoRA
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# ============================================================
# 3. Chat template + Data
# ============================================================
tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-instruct")

dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:1000]")
dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    return {
        "text": [
            tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
            for c in convos
        ]
    }
dataset = dataset.map(formatting_prompts_func, batched=True)

# ============================================================
# 4. Trainer
# ============================================================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,                      # demo; production: num_train_epochs=1
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

# ============================================================
# 5. Train
# ============================================================
trainer_stats = trainer.train()

# ============================================================
# 6. Inference
# ============================================================
from transformers import TextStreamer
messages = [{"role": "user", "content": "Faiz nedir?"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens = 500,
    temperature = 0.7, top_p = 0.8, top_k = 20,
    streamer = TextStreamer(tokenizer, skip_prompt=True),
)

# ============================================================
# 7. Save
# ============================================================
model.save_pretrained("qwen_lora")
tokenizer.save_pretrained("qwen_lora")

## Kaynaklar

**Resmi Unsloth notebook'lar (referans):**
- [Qwen3 4B Instruct](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)_Instruct.ipynb)
- [Qwen3 4B Thinking](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)_Thinking.ipynb)
- [Gemma 4 E2B Text](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(E2B)_Text.ipynb)
- [Qwen3.5 2B Vision](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb)

**Dokümanlar:**
- [Fine-tuning Guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide)
- [LoRA Hyperparameters](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide)
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer)

**Bu rehber neyi düzeltiyor (önceki yanlışlarım):**
- ❌ `assistant_only_loss=True` → ✅ `train_on_responses_only(...)` (Unsloth resmi)
- ❌ `processing_class=tokenizer` → ✅ `tokenizer=tokenizer` (resmi notebook'larda hala)
- ❌ `weight_decay=0.01` → ✅ `0.001`
- ❌ `lr_scheduler_type='cosine'` → ✅ `'linear'`
- ❌ `warmup_ratio=0.03` → ✅ `warmup_steps=5`
- ❌ `unsloth/Qwen3.5-2B-unsloth-bnb-4bit` (Hub'da YOK) → ✅ `unsloth/Qwen3.5-2B`
- ❌ `FastLanguageModel` legacy → ✅ Hala recommended (Qwen3 için)